# 🔬 D-FINE Single Line Diagram (SLD) Symbol Detection
### Interactive Training, Evaluation & DETR Box Refinement Notebook

This notebook configures and runs **D-FINE** (DETR architecture with Fine-Grained Distribution Refinement) directly on native COCO annotations.

### 1. Environment & Setup

In [ ]:
import os
import json
from pathlib import Path
import torch

base_dir = Path(".").resolve()
coco_dataset_dir = base_dir / "SLD ANNOTATION.v2-version-2.coco"
output_dir = base_dir / "results" / "dfine"
output_dir.mkdir(parents=True, exist_ok=True)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device.upper()}")
print(f"COCO Dataset Path: {coco_dataset_dir}")

### 2. Verify COCO Annotation Schema

In [ ]:
with open(coco_dataset_dir / "train" / "_annotations.coco.json", 'r') as f:
    coco_train = json.load(f)

print("Categories in COCO dataset:")
for cat in coco_train['categories']:
    print(f"  ID {cat['id']}: {cat['name']}")

print(f"\nTotal Train Images: {len(coco_train['images'])}")
print(f"Total Train Annotations: {len(coco_train['annotations'])}")

### 3. Generate D-FINE Custom Config

In [ ]:
dfine_config = {
    "task": "detection",
    "num_classes": 2,
    "remap_mscoco_category": False,
    "train_dataloader": {
        "img_folder": str(coco_dataset_dir / "train"),
        "ann_file": str(coco_dataset_dir / "train" / "_annotations.coco.json"),
        "batch_size": 2
    },
    "val_dataloader": {
        "img_folder": str(coco_dataset_dir / "test"),
        "ann_file": str(coco_dataset_dir / "test" / "_annotations.coco.json"),
        "batch_size": 2
    }
}

config_file = output_dir / "dfine_custom_sld.json"
with open(config_file, 'w') as f:
    json.dump(dfine_config, f, indent=2)
print(f"Created D-FINE config at: {config_file}")

### 4. Clone D-FINE Engine & Setup Dependencies

In [ ]:
dfine_repo = base_dir / "D-FINE"
if not dfine_repo.exists():
    print("Cloning D-FINE repository...")
    !git clone https://github.com/Peterande/D-FINE.git

# Install dependencies needed by D-FINE
!pip install -q pycocotools torchvision scipy onnx torchmetrics pyyaml

print("D-FINE repository and dependencies are ready!")

### 5. Configure D-FINE Custom Dataset YAML & Download Pretrained Weights

In [ ]:
import urllib.request
import yaml

# 1. Download base D-FINE weights if not present
weights_path = dfine_repo / "dfine_n_coco.pth"
if not weights_path.exists():
    print("Downloading D-FINE-N pretrained weights...")
    url = "https://github.com/Peterande/storage/releases/download/dfinev1.0/dfine_n_coco.pth"
    urllib.request.urlretrieve(url, str(weights_path))
    print(f"Saved weights to: {weights_path}")
else:
    print(f"Pretrained weights already present at: {weights_path}")

# 2. Write dynamic dataset YAML configuration for D-FINE
custom_dataset_yaml = dfine_repo / "configs" / "dataset" / "custom_detection.yml"
train_img = str((coco_dataset_dir / "train").resolve()).replace('\\', '/')
train_ann = str((coco_dataset_dir / "train" / "_annotations.coco.json").resolve()).replace('\\', '/')
val_img = str((coco_dataset_dir / "valid").resolve()).replace('\\', '/')
val_ann = str((coco_dataset_dir / "valid" / "_annotations.coco.json").resolve()).replace('\\', '/')

dataset_cfg = f"""task: detection

evaluator:
  type: CocoEvaluator
  iou_types: ['bbox', ]

num_classes: 2
remap_mscoco_category: False

train_dataloader:
  type: DataLoader
  dataset:
    type: CocoDetection
    img_folder: "{train_img}"
    ann_file: "{train_ann}"
    return_masks: False
    transforms:
      type: Compose
      ops: ~
  shuffle: True
  num_workers: 0
  drop_last: True
  collate_fn:
    type: BatchImageCollateFunction

val_dataloader:
  type: DataLoader
  dataset:
    type: CocoDetection
    img_folder: "{val_img}"
    ann_file: "{val_ann}"
    return_masks: False
    transforms:
      type: Compose
      ops: ~
  shuffle: False
  num_workers: 0
  drop_last: False
  collate_fn:
    type: BatchImageCollateFunction
"""

with open(custom_dataset_yaml, 'w') as f:
    f.write(dataset_cfg)

print(f"Configured {custom_dataset_yaml} successfully!")

### 6. Launch D-FINE Training

In [ ]:
import subprocess
import sys

print("Starting D-FINE Training...")

# Run training inside the D-FINE directory
%cd D-FINE
!python train.py -c configs/dfine/custom/dfine_hgnetv2_n_custom.yml -r dfine_n_coco.pth --use-amp --seed=0
%cd ..